# MiniMax H3 成熟工作流：Google Colab + Drive

本 Notebook 不公开私有 Director 节点。模型、环境快照、队列和成片都保存在 `MyDrive/minimaxH3_colab`。

私有源包已经预置到这个 Drive，第一次运行不需要 GitHub Token。请先把运行时切换到 GPU。首次会安装环境、下载约 43GB 模型并生成一次环境快照；后续会话直接恢复快照，不再重新安装整套依赖。以后若要同步私有源仓库的新版本，再临时添加 `GITHUB_TOKEN` 并启用重建开关。

In [ ]:
!nvidia-smi
import platform, sys
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/minimaxH3_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if token:
    os.environ['H3_SOURCE_GITHUB_TOKEN'] = token
    print('检测到 GITHUB_TOKEN：仅在首次构建或启用重建开关时刷新私有源包。')
else:
    print('使用 Drive 中预置的私有源包。')
print('Drive root:', DRIVE_ROOT)

In [ ]:
import shutil, subprocess
from pathlib import Path

LAUNCHER = Path('/content/minimaxH3_colab_launcher')
if LAUNCHER.exists():
    shutil.rmtree(LAUNCHER)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/SydneyMCDonaldbigking/minimaxH3_colab.git',
    str(LAUNCHER),
], check=True)

## 安装或恢复环境

第一次运行会构建 `/content/h3env + ComfyUI + 私有工作流`，验证通过后保存为 Drive 版本化快照。以后 Colab 重启时会恢复兼容快照，只重新挂接 Drive 模型并做校验。

默认只下载实际生产使用的 `ref2va` 模型。若以后同时需要纯文生，再把下面的 `ref2va` 改成 `both`。只有升级工作流、PyTorch 或 ComfyUI 时，才把 `REBUILD_RUNTIME` 改成 `True`。

In [ ]:
import subprocess, sys

REBUILD_RUNTIME = False
command = [
    sys.executable,
    '/content/minimaxH3_colab_launcher/colab/bootstrap_colab.py',
    '--drive-root', '/content/drive/MyDrive/minimaxH3_colab',
    '--profile', 'ref2va',
]
if REBUILD_RUNTIME:
    command.append('--rebuild-runtime')
subprocess.run(command, check=True)

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable,
    '/content/minimaxH3_colab_launcher/colab/launch_colab.py',
    '--drive-root', '/content/drive/MyDrive/minimaxH3_colab',
], check=True)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=900)

## 路径

- 通过机器 QA 的视频：`MyDrive/minimaxH3_colab/review_ready`
- 最终交付：`MyDrive/minimaxH3_colab/deliveries`
- 日志：`MyDrive/minimaxH3_colab/logs`
- 任务队列：`MyDrive/minimaxH3_colab/jobs`